<a href="https://www.kaggle.com/code/asivakumarnair/diabetic-retinopathy-imagenet?scriptVersionId=343468427" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ===== FULL REBUILD, NEW SESSION: environment through Stage 11, EYEPACS, MOBILENETV2 + RESNET50 =====

!pip install -q tensorflow==2.19.0

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import random
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.applications import MobileNetV2, ResNet50
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

# ---------- CONFIG ----------
EYEPACS_CSV   = '/kaggle/input/datasets/benjaminwarner/resized-2015-2019-blindness-detection-images/labels/trainLabels15.csv'
EYEPACS_IMG   = '/kaggle/input/datasets/benjaminwarner/resized-2015-2019-blindness-detection-images/resized train 15'

GRADES      = ['0','1','2','3','4']
IMG_SIZE, BATCH_SIZE = 224, 32
SUBSAMPLE_SEED = 42
EYEPACS_TARGET = 3662
PHASE1_EPOCHS, PHASE1_LR, PHASE2_LR, EARLYSTOP_PAT, MONITOR = 10, 1e-3, 1e-5, 7, 'val_accuracy'
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)

# ---------- DATA REBUILD, EYEPACS ONLY (needs patient-level split + subsample) ----------
eyepacs = pd.read_csv(EYEPACS_CSV)
eyepacs['grade']      = eyepacs['level'].astype(int).astype(str)
eyepacs['image_path'] = EYEPACS_IMG + '/' + eyepacs['image'].astype(str) + '.jpg'
eyepacs['source']     = 'eyepacs'
eyepacs['patient_id'] = eyepacs['image'].str.extract(r'^(\d+)_')
assert eyepacs['patient_id'].isna().sum() == 0, "EyePACS patient_id extraction failed"

def subsample_eyepacs(df, target_n=EYEPACS_TARGET, seed=SUBSAMPLE_SEED):
    pg = df.groupby('patient_id')['grade'].max().reset_index()
    frac = target_n / len(df)
    keep, _ = train_test_split(pg, train_size=frac, stratify=pg['grade'], random_state=seed)
    return df[df['patient_id'].isin(keep['patient_id'])].reset_index(drop=True)

eyepacs_s = subsample_eyepacs(eyepacs)
print(f"EyePACS subsampled: {len(eyepacs_s)} images, {eyepacs_s['patient_id'].nunique()} patients")

def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed, falling back to unstratified.")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_patient_level(df, rs=SEED, tag=""):
    pg = df.groupby('patient_id')['grade'].max().reset_index()
    p_tr, p_tmp = safe_split(pg, 'grade', 0.30, rs, tag=f"{tag} first")
    p_va, p_te  = safe_split(p_tmp, 'grade', 0.50, rs, tag=f"{tag} second")
    pick = lambda ids: df[df['patient_id'].isin(ids['patient_id'])]
    tr, va, te = pick(p_tr), pick(p_va), pick(p_te)
    s_tr, s_va, s_te = set(p_tr['patient_id']), set(p_va['patient_id']), set(p_te['patient_id'])
    assert s_tr.isdisjoint(s_va) and s_tr.isdisjoint(s_te) and s_va.isdisjoint(s_te), f"{tag} PATIENT LEAKAGE"
    print(f"{tag} patient-leakage check: PASS")
    return tr, va, te

e_tr, e_va, e_te = split_patient_level(eyepacs_s, tag="EyePACS")

cls = np.array(GRADES)
cw = compute_class_weight('balanced', classes=cls, y=e_tr['grade'])
eyepacs_class_weight = {i: w for i, w in enumerate(cw)}
span = cw.max()/cw.min()
print(f"\nEyePACS class weight span: {span:.1f}x (expect ~33.7x)")
print("This is roughly double the 16x collapse threshold observed elsewhere in this project.")
print("Watch AUC closely if accuracy/QWK looks broken, per Section 7.6 of the manual.")

# ================================================================
# STAGE 11: EYEPACS, MOBILENETV2 then RESNET50
# ================================================================

def make_source_gens(preprocess_fn, tr_df, va_df, te_df):
    train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
    eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='grade', target_size=(IMG_SIZE,IMG_SIZE),
                  batch_size=BATCH_SIZE, class_mode='categorical', classes=GRADES, color_mode='rgb')
    tr = train_idg.flow_from_dataframe(tr_df, shuffle=True,  seed=SEED, **common)
    va = eval_idg.flow_from_dataframe(va_df,  shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(te_df,  shuffle=False, **common)
    return tr, va, te

def build_pretrained(base_class, num_classes=5, shape=(224,224,3)):
    base = base_class(include_top=False, weights='imagenet', input_shape=shape)
    model = Sequential([base, GlobalAveragePooling2D(),
                         Dense(256,activation='relu'), Dropout(0.3),
                         Dense(num_classes,activation='softmax')])
    return model, base

def train_source_pretrained(base_class, preprocess_fn, arch_code, source_name, tr_df, va_df, te_df, class_weight):
    tag_p1 = f"ss_{arch_code}_{source_name}_dr_phase1"
    tag_p2 = f"ss_{arch_code}_{source_name}_dr"
    tr, va, te = make_source_gens(preprocess_fn, tr_df, va_df, te_df)
    model, base = build_pretrained(base_class)

    base.trainable = False
    model.compile(Adam(PHASE1_LR), 'categorical_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
    print(f"\n===== {arch_code}, {source_name}: PHASE 1 (head only, {PHASE1_EPOCHS} epochs) =====")
    model.fit(tr, validation_data=va, epochs=PHASE1_EPOCHS, class_weight=class_weight,
              callbacks=[ModelCheckpoint(f'/kaggle/working/{tag_p1}.keras', monitor=MONITOR, save_best_only=True),
                         CSVLogger(f'/kaggle/working/{tag_p1}_log.csv', append=False)], verbose=1)
    print(f"{arch_code}, {source_name} Phase 1 saved.")

    base.trainable = True
    model.compile(Adam(PHASE2_LR), 'categorical_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
    print(f"\n===== {arch_code}, {source_name}: PHASE 2 (full fine-tune, up to 60 epochs) =====")
    model.fit(tr, validation_data=va, epochs=60, class_weight=class_weight,
              callbacks=[EarlyStopping(monitor=MONITOR, patience=EARLYSTOP_PAT, restore_best_weights=True),
                         ModelCheckpoint(f'/kaggle/working/{tag_p2}.keras', monitor=MONITOR, save_best_only=True),
                         CSVLogger(f'/kaggle/working/{tag_p2}_log.csv', append=False)], verbose=1)
    result = model.evaluate(te, verbose=0)
    print(f"\n{tag_p2} TEST: loss={result[0]:.4f} accuracy={result[1]:.4f} auc={result[2]:.4f}")
    if result[2] > 0.75 and result[1] < 0.55:
        print(f"  ^ WATCH: AUC healthy but accuracy weak, consider val_auc fallback for a rerun if QWK also looks broken.")
    print(f"{arch_code}, {source_name} Phase 2 saved.")
    return {'arch':arch_code,'source':source_name,'loss':result[0],'accuracy':result[1],'auc':result[2]}

stage11_results = []

r = train_source_pretrained(MobileNetV2, mob_pre, 'mob', 'eyepacs', e_tr, e_va, e_te, eyepacs_class_weight)
stage11_results.append(r)
pd.DataFrame(stage11_results).to_csv('/kaggle/working/dr_stage11_eyepacs_mob.csv', index=False)
print("\nCheckpointed after MobileNetV2.")
tf.keras.backend.clear_session()

r = train_source_pretrained(ResNet50, res_pre, 'res', 'eyepacs', e_tr, e_va, e_te, eyepacs_class_weight)
stage11_results.append(r)
pd.DataFrame(stage11_results).to_csv('/kaggle/working/dr_stage11_eyepacs_mob_res.csv', index=False)
print("\nCheckpointed after ResNet50.")

print("\n===== EYEPACS, MobileNetV2 + ResNet50, COMPLETE =====")
print(pd.DataFrame(stage11_results).to_string(index=False))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 38.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.19.0 which is incompatible.
tf-keras 2.20.0 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.19.0 which is incompatible.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.19.0 which is incompatible.


2026-08-19 12:58:46.414547: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787144326.436788      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787144326.443867      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787144326.461620      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787144326.461638      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787144326.461641      23 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras
EyePACS subsampled: 3662 images, 1831 patients
EyePACS patient-leakage check: PASS

EyePACS class weight span: 33.7x (expect ~33.7x)
This is roughly double the 16x collapse threshold observed elsewhere in this project.
Watch AUC closely if accuracy/QWK looks broken, per Section 7.6 of the manual.
Found 2562 validated image filenames belonging to 5 classes.
Found 550 validated image filenames belonging to 5 classes.
Found 550 validated image filenames belonging to 5 classes.


I0000 00:00:1787144348.786395      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787144348.792458      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


9406464/9406464 [==============================] - 0s 0us/step

===== mob, eyepacs: PHASE 1 (head only, 10 epochs) =====
Epoch 1/10


I0000 00:00:1787144355.489944      76 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1787144358.255810      75 service.cc:152] XLA service 0x7fee3d2fa060 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1787144358.255859      75 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1787144358.255865      75 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1787144358.402989      75 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


81/81 [==============================] - 81s 922ms/step - loss: 1.9310 - accuracy: 0.2443 - auc: 0.6088 - val_loss: 1.6441 - val_accuracy: 0.1291 - val_auc: 0.5708
Epoch 2/10
81/81 [==============================] - 50s 623ms/step - loss: 1.4322 - accuracy: 0.2557 - auc: 0.6511 - val_loss: 1.3197 - val_accuracy: 0.4545 - val_auc: 0.7696
Epoch 3/10
81/81 [==============================] - 49s 603ms/step - loss: 1.3958 - accuracy: 0.3286 - auc: 0.7103 - val_loss: 1.6036 - val_accuracy: 0.2873 - val_auc: 0.6357
Epoch 4/10
81/81 [==============================] - 49s 609ms/step - loss: 1.2924 - accuracy: 0.3599 - auc: 0.7191 - val_loss: 1.2734 - val_accuracy: 0.5600 - val_auc: 0.7961
Epoch 5/10
81/81 [==============================] - 49s 608ms/step - loss: 1.2473 - accuracy: 0.3540 - auc: 0.7394 - val_loss: 1.3467 - val_accuracy: 0.4127 - val_auc: 0.7438
Epoch 6/10
81/81 [==============================] - 49s 608ms/step - loss: 1.1344 - accuracy: 0.3661 - auc: 0.7501 - val_loss: 1.5999 - 